# 01 · Data Research: BTC/USDT and ETH/USDT

Before building anything I want to understand how BTC and ETH actually behave.

What I'm checking here:
1. what data I have and whether I can trust it
2. return distributions: tails, skew, dependence
3. how volatility behaves and whether it can be forecast
4. time-of-day / day-of-week effects
5. regimes and structural changes
6. what volume, trade count and taker flow tell me
7. how BTC and ETH move together
8. which variables say anything about future returns or volatility

Rules I kept to:
- sections 1–2 (inventory, data quality) use all of 2021–2025, since checking quality doesn't bias anything
- from section 3 on I only use the train split (2021–2023); 2024 and 2025 stay untouched
- every predictor only uses data up to the close of its own bar

## 0. Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, acf, coint
from statsmodels.stats.diagnostic import acorr_ljungbox
from arch import arch_model

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 160)

In [ ]:
ROOT = Path.cwd().parent            # notebook lives in notebooks/
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

ASSETS = ["BTCUSDT", "ETHUSDT"]
COLORS = {"BTCUSDT": "#F7931A", "ETHUSDT": "#627EEA"}
TRAIN_END = "2023-12-31"


def savefig(fig, name):
    # Save a figure for the research report
    fig.savefig(FIG / f"{name}.png", dpi=120, bbox_inches="tight")

## 1. Data inventory

Listing the files, row counts and periods.

In [ ]:
rows = []
for path in sorted(RAW.glob("*.csv")) + sorted(PROC.glob("*/*.csv")):
    first_col = pd.read_csv(path, usecols=[0]).iloc[:, 0]
    rows.append({
        "file": str(path.relative_to(ROOT)),
        "rows": len(first_col),
        "first": first_col.iloc[0],
        "last": first_col.iloc[-1],
        "MB": round(path.stat().st_size / 1e6, 1),
    })
pd.DataFrame(rows)

Notes on the files:
- `data/raw/*_1h.csv`, `*_1d.csv`: Binance spot klines from `/api/v3/klines`, untouched
- `data/processed/1h`: cleaned hourly data (`src/data/preprocess.py`): types fixed, `close_time` dropped, 14 missing hours filled with flat candles, 2 empty outage candles flagged (`is_filled = True`), returns added
- `data/processed/4h`, `1d`: built from the cleaned hourly data, not downloaded

5 years = 1826 daily and 43,810 hourly raw rows per coin (43,824 after filling the gaps).

Looking at the raw columns next.

In [ ]:
raw = {s: pd.read_csv(RAW / f"{s}_1h.csv", parse_dates=["timestamp", "close_time"]) for s in ASSETS}
raw["BTCUSDT"].head(3)

| Column | Meaning |
|---|---|
| `timestamp` | candle **open** time, UTC |
| `open, high, low, close` | prices in USDT |
| `volume` | traded quantity in BTC / ETH |
| `close_time` | open + 59:59.999 |
| `quote_volume` | traded value in USDT |
| `trades` | number of trades in the hour |
| `taker_buy_base` | volume where the buyer was the aggressor |
| `taker_buy_quote` | same, in USDT |

A candle's close/high/low/volume are only known at `close_time`, one hour after `timestamp`. Checking that this holds everywhere.

In [ ]:
for s, df in raw.items():
    print(s, (df["close_time"] - df["timestamp"]).value_counts().to_dict())

43,806 of 43,810 candles close at :59:59. 4 per coin close early, checking which ones and what's in them.

In [ ]:
short = raw["BTCUSDT"]["close_time"] - raw["BTCUSDT"]["timestamp"] != pd.Timedelta("59min 59s")
raw["BTCUSDT"].loc[short, ["timestamp", "close_time", "volume", "trades"]]

Observations:
- the same 4 bars are short in both coins
- 2021-02-11 03:00 and 2023-03-24 12:00 have **0 volume and 0 trades**; the exchange stopped at :40 / :39, so they are really part of the outages after them. I flag them in preprocessing now (`trades == 0` → `is_filled`)
- 2021-04-25 04:00 has only 58 seconds of trading (5.9 BTC, 224 trades) before a 3-hour outage. Kept, but noted
- 2021-12-24 04:00 closes 5 seconds early, harmless

In [ ]:
def load(symbol, tf="1h", split="full"):
    return pd.read_csv(PROC / tf / f"{symbol}_{split}.csv", index_col="timestamp", parse_dates=True)


data = {s: load(s) for s in ASSETS}
data["BTCUSDT"].tail(3)

## 2. Data quality (full 2021–2025 period)

### 2.1 Gaps
Which hours are missing, and are they the same for both coins?

In [ ]:
filled = data["BTCUSDT"].index[data["BTCUSDT"]["is_filled"]].to_series()
new_block = filled.diff() != pd.Timedelta("1h")
outages = filled.groupby(new_block.cumsum()).agg(["first", "count"])
outages.columns = ["outage start", "hours missing"]

print("Same gaps in BTC and ETH:", data["BTCUSDT"]["is_filled"].equals(data["ETHUSDT"]["is_filled"]))
outages

### 2.2 Internal consistency
Checks on every real candle:
- high ≥ max(open, close), low ≤ min(open, close)
- taker buy volume ≤ volume
- VWAP (`quote_volume / volume`) inside [low, high]

In [ ]:
def consistency_checks(df):
    real = df[~df["is_filled"]]
    vwap = real["quote_volume"] / real["volume"]
    return pd.Series({
        "high < max(open, close)": (real["high"] < real[["open", "close"]].max(axis=1)).sum(),
        "low > min(open, close)": (real["low"] > real[["open", "close"]].min(axis=1)).sum(),
        "taker buy > volume": (real["taker_buy_base"] > real["volume"]).sum(),
        "zero volume": (real["volume"] == 0).sum(),
        "VWAP outside [low, high]": ((vwap < real["low"] * 0.9999) | (vwap > real["high"] * 1.0001)).sum(),
        "non-positive prices": (real[["open", "high", "low", "close"]] <= 0).sum().sum(),
    })


pd.DataFrame({s: consistency_checks(df) for s, df in data.items()})

### 2.3 Continuity
On a 24/7 market each open should equal the previous close. Checking for jumps.

In [ ]:
for s, df in data.items():
    jump = (df["open"] / df["close"].shift() - 1).abs()
    big = jump[jump > 0.001]
    after_outage = df["is_filled"].shift().reindex(big.index).fillna(False)
    print(f"{s}: median |open/prev close - 1| = {jump.median():.1e}, "
          f"jumps > 0.1%: {len(big)} (of which right after an outage: {after_outage.sum()})")

### 2.4 Extreme hourly moves
If a move is real it should show up in both coins and come with a volume spike. Checking the biggest ones.

In [ ]:
log_ret = pd.DataFrame({s: np.log(df["close"]).diff() for s, df in data.items()})
volume_x = pd.DataFrame({
    s: df["quote_volume"] / df["quote_volume"].rolling(24 * 30, min_periods=24).median()
    for s, df in data.items()
})

events = log_ret.abs().max(axis=1).nlargest(12).index.sort_values()
pd.DataFrame({
    "BTC ret %": log_ret.loc[events, "BTCUSDT"] * 100,
    "ETH ret %": log_ret.loc[events, "ETHUSDT"] * 100,
    "BTC volume x 30d median": volume_x.loc[events, "BTCUSDT"],
    "ETH volume x 30d median": volume_x.loc[events, "ETHUSDT"],
})

### 2.5 Wicks
A bad print would show up as a huge wick with a normal body. Listing the biggest wicks.

In [ ]:
def largest_wicks(df, n=5):
    body_top = df[["open", "close"]].max(axis=1)
    body_bottom = df[["open", "close"]].min(axis=1)
    wick = pd.concat([df["high"] / body_top - 1, 1 - df["low"] / body_bottom], axis=1).max(axis=1)
    out = df.loc[wick.nlargest(n).index, ["open", "high", "low", "close"]]
    out["wick %"] = wick.nlargest(n) * 100
    return out


for s, df in data.items():
    print(s)
    display(largest_wicks(df))

### 2.6 Trading activity through time
Binance ran a zero-fee promotion on BTC spot pairs (announced 8 Jul 2022, ended for BTCUSDT 22 Mar 2023). ETH had no promotion, so I use it as the comparison.

In [ ]:
ZERO_FEE = (pd.Timestamp("2022-07-08", tz="UTC"), pd.Timestamp("2023-03-22", tz="UTC"))

daily_activity = {s: df[["quote_volume", "trades"]].resample("1D").sum() for s, df in data.items()}

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for s, d in daily_activity.items():
    axes[0].plot(d.index, d["quote_volume"] / 1e9, color=COLORS[s], lw=0.8, label=s)
    axes[1].plot(d.index, d["trades"] / 1e6, color=COLORS[s], lw=0.8)
    axes[2].plot(d.index, d["quote_volume"] / d["trades"], color=COLORS[s], lw=0.8)
for ax, title in zip(axes, ["Daily volume (bn USDT)", "Daily trades (millions)", "Average trade size (USDT)"]):
    ax.axvspan(*ZERO_FEE, color="grey", alpha=0.2)
    ax.set_yscale("log")
    ax.set_title(title)
axes[0].legend()
fig.tight_layout()
savefig(fig, "activity_over_time")

In [ ]:
def period_label(ts):
    if ts < ZERO_FEE[0]:
        return "1 before"
    if ts < ZERO_FEE[1]:
        return "2 zero-fee"
    return "3 after"


rows = {}
for s, d in daily_activity.items():
    d = d.loc[:"2023-12-31"]
    g = d.groupby(d.index.map(period_label))
    rows[(s, "trades/day (M)")] = g["trades"].mean() / 1e6
    rows[(s, "volume/day (bn)")] = g["quote_volume"].mean() / 1e9
    rows[(s, "avg trade (USDT)")] = g["quote_volume"].sum() / g["trades"].sum()
pd.DataFrame(rows).T

### Observations: data quality
- **Gaps:** 7 outages, same hours in both coins. 14 missing hours + 2 empty candles = 16 flagged hours. Longest gap 4 h (2021-08-13)
- **Consistency:** 0 violations on every check (OHLC, taker ≤ volume, VWAP in range, prices > 0)
- **Continuity:** median |open / previous close − 1| is 9e-8 (BTC) and 2.6e-6 (ETH). Only 3 + 2 jumps above 0.1%, none right after an outage
- **Extreme moves look real:** all of them hit both coins in the same direction with 2–25× normal volume. Two are BTC-only: 2021-01-29 (+11.6% BTC vs +4.5% ETH) and 2021-02-08 (+8.6% vs +3.0%), the Musk bio change and the Tesla purchase news. The rest are market-wide crashes (2021-05-19, 2021-09-07, 2022-05-11 LUNA)
- **Big wicks are real liquidations:** BTC traded 30,000–38,000 inside one hour on 2021-05-19 (14.9% wick), 11.6% on 2021-12-04, 10.1% on 2025-10-10. ETH's biggest is 20.2% (2021-05-19)
- **Zero-fee period:** BTC trades/day went 1.63M → 6.26M → 1.04M (before / during / after), volume/day 2.8bn → 5.4bn → 1.2bn, average trade size 1,687 → 870 USDT. ETH over the same periods *fell*: 1.18M → 0.87M → 0.55M trades/day. So BTC's volume and trade counts change for exchange reasons, not only market reasons

## 3. Research sample: train split only (2021–2023)

From here on only 2021–2023. Log returns of closes, non-overlapping: 26,279 hourly, 6,569 4h, 1,094 daily, 156 weekly.

In [ ]:
train = {s: df.loc[:TRAIN_END] for s, df in data.items()}
close = pd.DataFrame({s: df["close"] for s, df in train.items()})
log_price = np.log(close)

HORIZONS = {"1h": "1h", "4h": "4h", "1d": "1D", "1w": "7D"}
PER_YEAR = {"1h": 24 * 365, "4h": 6 * 365, "1d": 365, "1w": 365 / 7}
rets = {h: log_price.resample(rule).last().diff().dropna() for h, rule in HORIZONS.items()}
r1h = rets["1h"]

{h: len(r) for h, r in rets.items()}

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for s in ASSETS:
    axes[0].plot(close.index, close[s] / close[s].iloc[0], color=COLORS[s], lw=0.8, label=s)
axes[0].set_yscale("log")
axes[0].set_title("Price, normalised to 1 on 2021-01-01 (log scale)")
axes[0].legend()
axes[1].plot(close.index, close["ETHUSDT"] / close["BTCUSDT"], color="black", lw=0.8)
axes[1].set_title("ETH/BTC ratio")
fig.tight_layout()
savefig(fig, "prices")

## 4. Return distributions

### 4.1 Summary statistics by horizon

In [ ]:
def describe_returns(r, per_year):
    return pd.Series({
        "obs": len(r),
        "ann. mean %": r.mean() * per_year * 100,
        "ann. vol %": r.std() * np.sqrt(per_year) * 100,
        "skew": r.skew(),
        "excess kurtosis": r.kurt(),
        "min %": r.min() * 100,
        "max %": r.max() * 100,
        "Jarque-Bera p": stats.jarque_bera(r).pvalue,
    })


pd.DataFrame({(s, h): describe_returns(r[s], PER_YEAR[h]) for h, r in rets.items() for s in ASSETS})

### 4.2 How fat are the tails?
Comparing how often |z| > k actually happens with what a normal distribution would give.

In [ ]:
def tail_ratio(r):
    z = (r - r.mean()) / r.std()
    return pd.Series({f"|z| > {k}": (z.abs() > k).mean() / (2 * stats.norm.sf(k)) for k in [3, 4, 5, 6]})


pd.DataFrame({(s, h): tail_ratio(rets[h][s]) for h in ["1h", "1d"] for s in ASSETS})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, s in zip(axes, ASSETS):
    z = (r1h[s] - r1h[s].mean()) / r1h[s].std()
    bins = np.linspace(-12, 12, 121)
    ax.hist(z, bins=bins, density=True, color=COLORS[s], alpha=0.7, label="observed")
    ax.plot(bins, stats.norm.pdf(bins), "k--", label="normal")
    ax.set_yscale("log")
    ax.set_ylim(1e-6, 1)
    ax.set_title(f"{s} hourly returns (standardised)")
    ax.legend()
fig.tight_layout()
savefig(fig, "return_distribution")

### 4.3 Tail index (Hill estimator)
P(|r| > x) ~ x^(−α). α < 4 means kurtosis isn't really defined; α < 2 would mean infinite variance.
The estimate depends on how much of the tail I use, so I tried 1%, 2.5% and 5%.

In [ ]:
def hill_alpha(x, tail_frac):
    x = np.sort(x[x > 0])[::-1]
    k = int(len(x) * tail_frac)
    return 1 / np.mean(np.log(x[:k] / x[k]))


rows = {}
for s in ASSETS:
    for h in ["1h", "1d"]:
        r = rets[h][s].values
        for frac in [0.01, 0.025, 0.05]:
            rows[(s, h, frac)] = {"left tail α": hill_alpha(-r, frac), "right tail α": hill_alpha(r, frac)}
pd.DataFrame(rows).T

### 4.4 How much of the fat tail is just volatility clustering?
Dividing each hourly return by a past-only volatility estimate (EWMA, 1-week span, shifted one bar) and checking how much kurtosis is left.

In [ ]:
ewma_vol = np.sqrt((r1h ** 2).ewm(span=24 * 7).mean().shift(1))
z_scaled = (r1h / ewma_vol).dropna()

pd.DataFrame({
    "raw excess kurtosis": r1h.kurt(),
    "vol-scaled excess kurtosis": z_scaled.kurt(),
    "raw skew": r1h.skew(),
    "vol-scaled skew": z_scaled.skew(),
})

In [ ]:
rows = {}
for s in ASSETS:
    nu, loc, scale = stats.t.fit(rets["1d"][s])
    rows[s] = {"Student-t degrees of freedom (daily)": nu}
pd.DataFrame(rows)

Kurtosis went **up** after scaling (BTC 15.7 → 24.7, ETH 13.4 → 18.0). Looking at which hours give the biggest scaled shocks.

In [ ]:
pd.DataFrame({
    "vol-scaled z": z_scaled["BTCUSDT"],
    "return %": r1h["BTCUSDT"] * 100,
    "previous hourly vol %": ewma_vol["BTCUSDT"] * 100,
}).loc[z_scaled["BTCUSDT"].abs().nlargest(8).index]

In [ ]:
# Sensitivity: different EWMA spans, and removing just the 5 largest shocks
rows = {}
for span in [24, 168, 720]:
    vol = np.sqrt((r1h ** 2).ewm(span=span).mean().shift(1))
    rows[f"EWMA span {span}h"] = (r1h / vol).kurt()
z_btc = z_scaled["BTCUSDT"]
print("BTC vol-scaled kurtosis without its 5 largest shocks:", round(z_btc.drop(z_btc.abs().nlargest(5).index).kurt(), 1))
pd.DataFrame(rows)

### Observations: distributions
- **Vol level:** BTC ~65% a year, ETH ~85%, almost the same at every horizon (BTC 66.5% hourly vs 64.9% daily)
- **Fat tails:** hourly excess kurtosis 15.7 (BTC) and 13.4 (ETH). |z| > 4 happens ~135× more often than normal. Daily kurtosis drops to 3.5 / 5.7 but is still fat at weekly (3.1 / 3.5)
- **Negative skew**, stronger for ETH: −0.59 hourly vs −0.19 for BTC
- **Tail index** ≈ 2.6–3.7 hourly; Student-t degrees of freedom on daily returns 2.5 (BTC) and 2.9 (ETH). Variance finite, 4th moment probably not, so the kurtosis values themselves are unstable
- **Biggest shocks come out of calm markets:** after vol-scaling, the top shocks are ~5% hourly moves when hourly vol was only ~0.25%, mostly in quiet 2023 (2023-08-29 +5.4% at z = 21, 2023-08-17 −5.3% at z = −19). Dropping just the 5 largest takes BTC's scaled kurtosis from 24.7 to 12.8. Same picture for every EWMA span (24h / 168h / 720h)

## 5. Stationarity

- ADF: null = unit root. Small p → stationary
- KPSS: null = stationary. Small p → non-stationary

Using both since each one alone is weak. Neither checks for constant variance (section 7 does).

In [ ]:
def stationarity(x):
    return pd.Series({
        "ADF p": adfuller(x, autolag="AIC")[1],
        "KPSS p": kpss(x, regression="c", nlags="auto")[1],
    })


daily_log_price = log_price.resample("1D").last()
rows = {}
for s in ASSETS:
    rows[(s, "log price")] = stationarity(daily_log_price[s])
    rows[(s, "daily log return")] = stationarity(rets["1d"][s])
pd.DataFrame(rows).T

### Observations: stationarity
- log prices: ADF p = 0.55 (BTC) / 0.20 (ETH), KPSS p = 0.01 → non-stationary
- daily log returns: ADF p = 0.00, KPSS p = 0.10 → stationary in the mean
- the variance of returns clearly isn't constant (section 7)

## 6. Serial dependence: do past returns tell us about future returns?

### 6.1 Autocorrelation of returns vs absolute returns
The dashed ±1.96/√n band is too narrow for data with changing volatility, so small spikes in the return ACF don't mean much.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
for s in ASSETS:
    axes[0].plot(range(1, 49), acf(r1h[s], nlags=48)[1:], "o-", ms=3, color=COLORS[s], label=s)
    axes[1].plot(range(1, 24 * 14 + 1), acf(r1h[s].abs(), nlags=24 * 14)[1:], color=COLORS[s], label=s)
band = 1.96 / np.sqrt(len(r1h))
axes[0].axhline(band, ls="--", c="k", lw=0.8)
axes[0].axhline(-band, ls="--", c="k", lw=0.8)
axes[0].set_title("ACF of hourly returns (lags 1–48 h)")
axes[1].set_title("ACF of absolute hourly returns (lags 1 h – 2 weeks)")
axes[1].set_xlabel("lag (hours)")
for ax in axes:
    ax.axhline(0, c="k", lw=0.5)
    ax.legend()
fig.tight_layout()
savefig(fig, "acf")

### 6.2 Ljung-Box test (24 lags)
On raw returns, vol-scaled returns and squared returns.

In [ ]:
rows = {}
for s in ASSETS:
    series = {"returns": r1h[s], "vol-scaled returns": z_scaled[s], "squared returns": r1h[s] ** 2}
    rows[s] = {name: acorr_ljungbox(x, lags=[24])["lb_pvalue"].iloc[0] for name, x in series.items()}
pd.DataFrame(rows).T

### 6.3 Is the lag-1 autocorrelation stable across years?

In [ ]:
rows = {}
for h in ["1h", "4h", "1d"]:
    for s in ASSETS:
        r = rets[h][s]
        rows[(s, h)] = r.groupby(r.index.year).apply(lambda x: x.autocorr(1))
lag1 = pd.DataFrame(rows).T
lag1["±2 s.e. (1 yr)"] = [2 / np.sqrt(len(rets[h]) / 3) for h in ["1h", "1h", "4h", "4h", "1d", "1d"]]
lag1

### 6.4 Variance ratio test
VR(q) = Var(q-period return) / (q · Var(1-period return)).
- VR > 1 → trending
- VR < 1 → mean-reverting
- VR = 1 → random walk

Using the Lo–MacKinlay z that allows for changing volatility.

In [ ]:
def variance_ratio(r, q):
    r = r.dropna().values
    mu = r.mean()
    e2 = (r - mu) ** 2
    var1 = e2.sum() / (len(r) - 1)
    rq = np.convolve(r, np.ones(q), mode="valid")          # overlapping q-period sums
    m = q * (len(r) - q + 1) * (1 - q / len(r))
    vr = ((rq - q * mu) ** 2).sum() / m / var1

    theta = 0.0                                              # heteroskedasticity-robust variance
    for j in range(1, q):
        delta = (e2[j:] * e2[:-j]).sum() / e2.sum() ** 2
        theta += (2 * (q - j) / q) ** 2 * delta
    return vr, (vr - 1) / np.sqrt(theta)

In [ ]:
rows = {}
for s in ASSETS:
    for q in [2, 4, 12, 24, 72, 168]:
        vr, z = variance_ratio(r1h[s], q)
        rows[(s, "hourly", q)] = {"VR": vr, "z": z}
    for q in [2, 5, 10, 20]:
        vr, z = variance_ratio(rets["1d"][s], q)
        rows[(s, "daily", q)] = {"VR": vr, "z": z}
pd.DataFrame(rows).T

In [ ]:
# Same test, one year at a time: does the sign of the effect stay the same?
rows = {}
for s in ASSETS:
    for year in [2021, 2022, 2023]:
        r = r1h[s].loc[str(year)]
        rows[(s, year)] = {f"VR({q}h)": variance_ratio(r, q)[0] for q in [2, 24, 168]}
pd.DataFrame(rows).T

### 6.5 Pearson vs rank (Spearman) autocorrelation
Pearson is dominated by the few huge moves, Spearman weighs every hour equally.

In [ ]:
pd.DataFrame({
    s: {
        "Pearson lag-1": r1h[s].autocorr(1),
        "Spearman lag-1": r1h[s].corr(r1h[s].shift(1), method="spearman"),
    }
    for s in ASSETS
})

### Observations: serial dependence
- **Returns** have ≈ 0 autocorrelation; **absolute returns** stay autocorrelated for weeks, with bumps every 24 h
- **Ljung-Box** p ≈ 0 for returns, vol-scaled returns (0.0006 BTC) and squared returns: some linear structure, but small
- **Lag-1 isn't stable:** hourly +0.003 / +0.003 / −0.041 for BTC in 2021 / 2022 / 2023 (±2 s.e. = 0.021)
- **Variance ratios ≈ 1** everywhere (0.90–1.02), no |z| above 1.4
- **Year by year it flips:** BTC weekly VR(168h) = 0.83 in 2021, 1.05 in 2022, 1.34 in 2023. ETH 0.85 / 1.18 / 0.92
- **Pearson vs rank:** lag-1 Pearson −0.003 (BTC) / +0.011 (ETH), Spearman −0.051 / −0.038. Normal-sized moves slightly reverse, the huge ones don't (followed up in section 12)

## 7. Volatility

### 7.1 Volatility through time
30-day realised vol from hourly returns, annualised.

In [ ]:
rv_30d = np.sqrt((r1h ** 2).rolling(24 * 30).sum() * 365 / 30) * 100

fig, ax = plt.subplots()
for s in ASSETS:
    ax.plot(rv_30d.index, rv_30d[s], color=COLORS[s], lw=0.9, label=s)
ax.set_title("30-day realised volatility (annualised, %)")
ax.legend()
savefig(fig, "realised_vol")

rv_30d.describe().T[["min", "50%", "max"]]

### 7.2 Daily realised variance and its persistence
Daily RV = sum of the 24 squared hourly returns.

In [ ]:
rv_daily = (r1h ** 2).resample("1D").sum()
log_rv = np.log(rv_daily)

fig, ax = plt.subplots()
for s in ASSETS:
    ax.plot(range(1, 91), acf(log_rv[s], nlags=90)[1:], color=COLORS[s], label=s)
ax.axhline(0, c="k", lw=0.5)
ax.set_title("ACF of daily log realised variance")
ax.set_xlabel("lag (days)")
ax.legend()
savefig(fig, "rv_persistence")

pd.DataFrame({s: {f"ACF lag {k}d": acf(log_rv[s], nlags=90)[k] for k in [1, 7, 30, 90]} for s in ASSETS})

### 7.3 Is volatility forecastable? HAR model, out of sample
tomorrow's log RV = a + b·(today) + c·(last 7 days) + d·(last 30 days).
Fit on 2021–2022, test on 2023, compared with "tomorrow = today" and "tomorrow = 2021–22 average".

In [ ]:
def har_frame(lrv):
    return pd.DataFrame({
        "day": lrv,
        "week": lrv.rolling(7).mean(),
        "month": lrv.rolling(30).mean(),
        "next_day": lrv.shift(-1),
    }).dropna()


rows = {}
for s in ASSETS:
    f = har_frame(log_rv[s])
    fit, test = f.loc[:"2022"], f.loc["2023"]
    X_cols = ["day", "week", "month"]
    model = sm.OLS(fit["next_day"], sm.add_constant(fit[X_cols])).fit()
    pred = model.predict(sm.add_constant(test[X_cols]))

    mse = lambda p: ((test["next_day"] - p) ** 2).mean()
    rows[s] = {
        "in-sample R²": model.rsquared,
        "2023 R² (vs 2021-22 mean)": 1 - mse(pred) / mse(fit["next_day"].mean()),
        "2023 MSE / naive 'same as today'": mse(pred) / mse(test["day"]),
        "b day": model.params["day"], "c week": model.params["week"], "d month": model.params["month"],
    }
pd.DataFrame(rows)

### 7.4 GARCH: persistence and asymmetry
GJR-GARCH(1,1) with Student-t errors on daily returns. Positive gamma would mean down moves raise vol more than up moves (the stock-market "leverage effect").

In [ ]:
rows = {}
for s in ASSETS:
    res = arch_model(rets["1d"][s] * 100, mean="Constant", vol="GARCH", p=1, o=1, q=1, dist="t").fit(disp="off")
    a, g, b = res.params["alpha[1]"], res.params["gamma[1]"], res.params["beta[1]"]
    persistence = a + b + g / 2
    rows[s] = {
        "alpha": a, "gamma": g, "gamma p-value": res.pvalues["gamma[1]"], "beta": b,
        "persistence": persistence,
        # half-life only exists if persistence < 1; at ~1 shocks never fade (integrated GARCH)
        "half-life (days)": np.log(0.5) / np.log(persistence) if persistence < 0.999 else np.inf,
        "t dof": res.params["nu"],
    }
pd.DataFrame(rows)

### 7.5 Do high/low prices add volatility information?
Correlating three estimates of today's variance with tomorrow's RV: squared close-to-close return, Parkinson (high/low), hourly RV.

In [ ]:
rows = {}
for s in ASSETS:
    d1 = load(s, "1d").loc[:TRAIN_END]
    estimators = pd.DataFrame({
        "squared daily return": np.log(d1["close"]).diff() ** 2,
        "Parkinson (high/low)": np.log(d1["high"] / d1["low"]) ** 2 / (4 * np.log(2)),
        "hourly RV": rv_daily[s],
    })
    next_rv = np.log(rv_daily[s]).shift(-1)
    rows[s] = np.log(estimators.replace(0, np.nan)).corrwith(next_rv, method="spearman")
pd.DataFrame(rows)

### Observations: volatility
- **30-day vol range:** BTC 22%–135% (median 60%), ETH 22%–193% (median 74%). Average level roughly halved from 2021 to 2023 (BTC 81% → 44%, ETH 109% → 47%)
- **Persistence:** autocorrelation of daily log RV is 0.63 / 0.71 at 1 day and still 0.26 / 0.33 at 90 days (BTC / ETH)
- **HAR out of sample (2023):** R² 0.60 (BTC) and 0.73 (ETH), error 34% lower than "tomorrow = today" (MSE ratio 0.66 / 0.67). The weekly term has the biggest weight (0.51 / 0.47)
- **GARCH persistence = 1.00** for both, half-life infinite. Probably a sign the vol *level* shifts over the period rather than shocks never fading
- **No leverage effect:** gamma −0.001 (p = 0.95) for BTC and 0.008 (p = 0.68) for ETH
- **High/low helps:** correlation with tomorrow's RV is 0.38 for the squared daily return, 0.58 for Parkinson, 0.62 for hourly RV (BTC). ETH 0.42 / 0.65 / 0.69

## 8. Intraday and weekly seasonality

### 8.1 Hour of day (UTC)

In [ ]:
hour = r1h.index.hour
volume_share = {s: train[s]["quote_volume"] / train[s]["quote_volume"].resample("1D").transform("sum") for s in ASSETS}

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for s in ASSETS:
    axes[0].plot(r1h[s].abs().groupby(hour).mean() * 100, "o-", color=COLORS[s], label=s)
    axes[1].plot(volume_share[s].groupby(volume_share[s].index.hour).mean() * 100, "o-", color=COLORS[s])
    g = r1h[s].groupby(hour)
    mean, se = g.mean() * 1e4, g.std() / np.sqrt(g.count()) * 1e4
    axes[2].errorbar(mean.index + (0.15 if s == "ETHUSDT" else 0), mean, yerr=1.96 * se, fmt="o", color=COLORS[s])
axes[0].set_title("Mean |hourly return| (%)")
axes[1].set_title("Share of daily volume (%)")
axes[2].set_title("Mean hourly return (bps) with 95% CI")
axes[2].axhline(0, c="k", lw=0.5)
axes[2].set_xlabel("hour of day (UTC)")
axes[2].set_xticks(range(24))
axes[0].legend()
fig.tight_layout()
savefig(fig, "hour_of_day")

In [ ]:
# Busiest and quietest hours (UTC) by average absolute return, relative to the daily average
for s in ASSETS:
    prof = r1h[s].abs().groupby(hour).mean()
    prof = prof / prof.mean()
    print(s, "most volatile:", prof.nlargest(3).round(2).to_dict(), "| quietest:", prof.nsmallest(3).round(2).to_dict())

Stability check: building the 24-hour profile separately for 2021, 2022 and 2023 and correlating them.
If an effect is real the profiles should match across years.

In [ ]:
def profile_correlation(x):
    profile = x.groupby([x.index.year, x.index.hour]).mean().unstack(0)
    c = profile.corr().values
    return c[np.triu_indices(3, k=1)].mean()


pd.DataFrame({
    s: {
        "|return| profile: mean corr between years": profile_correlation(r1h[s].abs()),
        "return profile: mean corr between years": profile_correlation(r1h[s]),
        "hours with |t| > 2 (return)": (np.abs(r1h[s].groupby(hour).mean() / (r1h[s].groupby(hour).std() / np.sqrt(r1h[s].groupby(hour).count()))) > 2).sum(),
        "expected by chance (5% of 24)": 1.2,
    }
    for s in ASSETS
})

### 8.2 Day of week

In [ ]:
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
d1 = rets["1d"]
rows = {}
for s in ASSETS:
    g = d1[s].groupby(d1.index.dayofweek)
    daily_volume = train[s]["quote_volume"].resample("1D").sum()
    rows[(s, "mean |ret| %")] = d1[s].abs().groupby(d1.index.dayofweek).mean() * 100
    rows[(s, "mean ret %")] = g.mean() * 100
    rows[(s, "t-stat")] = g.mean() / (g.std() / np.sqrt(g.count()))
    rows[(s, "volume vs avg")] = daily_volume.groupby(daily_volume.index.dayofweek).mean() / daily_volume.mean()
weekday = pd.DataFrame(rows)
weekday.index = day_names
weekday

### Observations: seasonality
- **Volatility follows the clock:** busiest hours 14 UTC (1.31× average for BTC), 15 UTC (1.21×) and 00 UTC (1.20×); quietest 04–05 UTC (0.74× / 0.79×). The |return| profiles correlate 0.65 (BTC) and 0.71 (ETH) between years
- **Returns don't:** return profiles correlate −0.16 between years, and only 1 hour of 24 has |t| > 2, about what chance gives (1.2)
- **Weekends are quiet:** weekend volume ~0.72–0.77× average, Saturday's average |move| 1.25% vs 2.4–3.0% on weekdays (BTC). No weekday has a significant mean return (all |t| < 1.1)

## 9. Regimes and structural change

### 9.1 Year by year

In [ ]:
def max_drawdown(price):
    return (price / price.cummax() - 1).min()


rows = {}
for s in ASSETS:
    for year in [2021, 2022, 2023]:
        r = rets["1d"][s].loc[str(year)]
        p = close[s].loc[str(year)]
        rows[(s, year)] = {
            "return %": (np.exp(r.sum()) - 1) * 100,
            "ann. vol %": r.std() * np.sqrt(365) * 100,
            "skew": r.skew(),
            "excess kurt": r.kurt(),
            "max drawdown %": max_drawdown(p) * 100,
            "daily lag-1 autocorr": r.autocorr(1),
            "avg daily volume (bn USDT)": train[s]["quote_volume"].loc[str(year)].resample("1D").sum().mean() / 1e9,
        }
pd.DataFrame(rows).T

### 9.2 Buy-and-hold drawdowns
How deep buy-and-hold went over 2021–2023.

In [ ]:
fig, ax = plt.subplots()
rows = {}
for s in ASSETS:
    dd = close[s] / close[s].cummax() - 1
    ax.fill_between(dd.index, dd * 100, 0, color=COLORS[s], alpha=0.4, label=s)
    underwater = (dd < 0).astype(int)
    longest = underwater.groupby((underwater == 0).cumsum()).sum().max() / 24
    rows[s] = {"max drawdown %": dd.min() * 100, "date of max drawdown": dd.idxmin().date(), "longest underwater (days)": longest}
ax.set_title("Buy-and-hold drawdown (%)")
ax.legend()
savefig(fig, "drawdowns")
pd.DataFrame(rows)

### 9.3 Volatility regimes
Each day labelled low / mid / high vol from **yesterday's** 30-day realised vol.
The tercile cut-offs use the whole train period, so this is descriptive only.

In [ ]:
daily_rv30 = np.sqrt(rv_daily.rolling(30).sum() * 365 / 30).shift(1)
d1 = rets["1d"]

rows = {}
for s in ASSETS:
    regime = pd.qcut(daily_rv30[s], 3, labels=["low vol", "mid vol", "high vol"])
    frame = pd.DataFrame({"ret": d1[s], "btc": d1["BTCUSDT"], "eth": d1["ETHUSDT"], "regime": regime}).dropna()
    for name, g in frame.groupby("regime", observed=True):
        rows[(s, name)] = {
            "days": len(g),
            "mean daily ret %": g["ret"].mean() * 100,
            "t-stat": g["ret"].mean() / g["ret"].std() * np.sqrt(len(g)),
            "daily vol %": g["ret"].std() * 100,
            "lag-1 autocorr": g["ret"].autocorr(1),
            "BTC-ETH corr": g["btc"].corr(g["eth"]),
        }
pd.DataFrame(rows).T

### 9.4 Trend regimes
Each day labelled above / below its 200-day moving average (known at the close), then looking at the **next** day.

In [ ]:
daily_close = close.resample("1D").last()
rows = {}
for s in ASSETS:
    above = (daily_close[s] > daily_close[s].rolling(200).mean()).where(daily_close[s].rolling(200).count() == 200)
    frame = pd.DataFrame({"next_ret": d1[s].shift(-1), "above": above}).dropna()
    for flag, g in frame.groupby("above"):
        rows[(s, "above 200d MA" if flag else "below 200d MA")] = {
            "days": len(g),
            "mean next-day ret %": g["next_ret"].mean() * 100,
            "t-stat": g["next_ret"].mean() / g["next_ret"].std() * np.sqrt(len(g)),
            "next-day vol %": g["next_ret"].std() * 100,
        }
pd.DataFrame(rows).T

### Observations: regimes
- **Three different years:** 2021 BTC +58% / ETH +404%, 2022 −64% / −67%, 2023 +156% / +91%. BTC vol fell from 81% to 44%, and its daily skew went −0.06 (2021), −0.58 (2022), +0.70 (2023)
- **Buy-and-hold drawdown:** −77.2% (BTC, bottom 2022-11-21) and −81.3% (ETH, bottom 2022-06-18), ~781 days underwater for both
- **Vol regimes vs direction:** no significant mean return difference (all |t| < 1.5). BTC–ETH correlation slightly lower in high vol (0.82 vs 0.86). BTC daily lag-1 autocorrelation in high vol is −0.14, the only number that stands out
- **200-day MA:** next-day returns above / below are not significantly different (|t| ≤ 1.1), but next-day vol is lower above the MA: 2.7% vs 3.3% (BTC), 3.3% vs 4.2% (ETH)

## 10. Volume, trades and order-flow variables

Hourly variables from each real candle:
- `range`: log(high/low)
- `log_volume`, `log_trades`
- `avg_trade`: USDT per trade
- `taker_buy_ratio`: share of volume from aggressive buyers (0.5 = balanced)
- `amihud`: |return| per $1M traded

In [ ]:
def bar_variables(df):
    real = df[~df["is_filled"] & (df["trades"] > 0)]         # also drop the hidden outage bars
    ret = np.log(df["close"]).diff().loc[real.index]
    return pd.DataFrame({
        "ret": ret,
        "abs_ret": ret.abs(),
        "range": np.log(real["high"] / real["low"]),
        "log_volume": np.log(real["quote_volume"]),
        "log_trades": np.log(real["trades"]),
        "avg_trade": real["quote_volume"] / real["trades"],
        "taker_buy_ratio": real["taker_buy_base"] / real["volume"],
        "amihud": ret.abs() / real["quote_volume"] * 1e6,
    })


bars = {s: bar_variables(df) for s, df in train.items()}
bars["BTCUSDT"].describe().T

### 10.1 Which variables are redundant?
Spearman correlations between variables in the same hour.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, s in zip(axes, ASSETS):
    sns.heatmap(bars[s].corr(method="spearman"), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax, cbar=False)
    ax.set_title(s)
fig.tight_layout()
savefig(fig, "variable_correlations")

bars["BTCUSDT"].corr(method="spearman").round(2)

### 10.2 Taker buy ratio: same hour vs next hour
It should be tied to the same hour's return (aggressive buying pushes price up). Checking the next hour too.

In [ ]:
rows = {}
for s in ASSETS:
    b = bars[s]
    next_ret = b["ret"].shift(-1)
    rows[s] = {
        "corr with same-hour return": b["taker_buy_ratio"].corr(b["ret"], method="spearman"),
        "corr with next-hour return": b["taker_buy_ratio"].corr(next_ret, method="spearman"),
        "mean ratio": b["taker_buy_ratio"].mean(),
    }
pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots()
for s in ASSETS:
    daily_ratio = train[s]["taker_buy_base"].resample("1D").sum() / train[s]["volume"].resample("1D").sum()
    ax.plot(daily_ratio.rolling(7).mean(), color=COLORS[s], lw=0.9, label=s)
ax.axvspan(*ZERO_FEE, color="grey", alpha=0.2, label="BTC zero-fee period")
ax.axhline(0.5, c="k", lw=0.5)
ax.set_title("Taker buy ratio (7-day average)")
ax.legend()
savefig(fig, "taker_ratio")

In [ ]:
# Average taker buy ratio before / during / after the BTC zero-fee period
pd.DataFrame({
    s: bars[s]["taker_buy_ratio"].groupby(bars[s].index.map(period_label)).mean()
    for s in ASSETS
})

### Observations: volume and order flow
- `log_volume` and `log_trades` are 0.92 rank-correlated; `range` vs `abs_ret` 0.73, `range` vs volume 0.62. Mostly measuring the same thing: how busy the hour is
- **Taker buy ratio:** 0.49 (BTC) / 0.51 (ETH) with the same hour's return, −0.044 / −0.025 with the next hour
- **Taker ratio vs fees:** BTC average 0.493 before, 0.498 during and 0.482 after the zero-fee period; ETH 0.499 / 0.499 / 0.493
- **Amihud** is 0.68 correlated with |return|

## 11. BTC and ETH together

### 11.1 Correlation by horizon

In [ ]:
pd.Series({h: r["BTCUSDT"].corr(r["ETHUSDT"]) for h, r in rets.items()}, name="BTC-ETH correlation")

### 11.2 Is the relationship stable?
Rolling 30-day correlation and beta (ETH move per 1% BTC move) on hourly returns.

In [ ]:
window = 24 * 30
roll_corr = r1h["BTCUSDT"].rolling(window).corr(r1h["ETHUSDT"])
roll_beta = r1h["BTCUSDT"].rolling(window).cov(r1h["ETHUSDT"]) / r1h["BTCUSDT"].rolling(window).var()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(roll_corr, color="purple", lw=0.9)
axes[0].set_title("Rolling 30-day correlation (hourly returns)")
axes[1].plot(roll_beta, color="teal", lw=0.9)
axes[1].axhline(1, c="k", lw=0.5)
axes[1].set_title("Rolling 30-day beta of ETH on BTC")
fig.tight_layout()
savefig(fig, "rolling_corr_beta")

pd.DataFrame({"correlation": roll_corr, "beta": roll_beta}).describe().T[["mean", "min", "25%", "75%", "max"]]

### 11.3 Lead-lag
Correlation of BTC's return in hour *t* with ETH's in hour *t + k*. Positive k = BTC first.

In [ ]:
lead_lag = pd.Series({k: r1h["BTCUSDT"].corr(r1h["ETHUSDT"].shift(-k)) for k in range(-3, 4)}, name="corr(BTC_t, ETH_t+k)")
print("approx. 2 s.e.:", round(2 / np.sqrt(len(r1h)), 4))
lead_lag

### 11.4 Tail dependence
P(ETH in its worst q% | BTC in its worst q%), against a bivariate normal with the same correlation.

In [ ]:
def co_exceedance(x, y, q):
    lower = ((x <= np.quantile(x, q)) & (y <= np.quantile(y, q))).mean() / q
    upper = ((x >= np.quantile(x, 1 - q)) & (y >= np.quantile(y, 1 - q))).mean() / q
    return lower, upper


rng = np.random.default_rng(0)
rows = {}
for h in ["1h", "1d"]:
    x, y = rets[h]["BTCUSDT"].values, rets[h]["ETHUSDT"].values
    rho = np.corrcoef(x, y)[0, 1]
    sim = rng.multivariate_normal([0, 0], [[1, rho], [rho, 1]], size=1_000_000)
    for q in [0.05, 0.01]:
        lo, up = co_exceedance(x, y, q)
        lo_n, up_n = co_exceedance(sim[:, 0], sim[:, 1], q)
        rows[(h, q)] = {"crash together": lo, "normal would give": lo_n, "rally together": up, "normal would give ": up_n}
pd.DataFrame(rows).T

### 11.5 The ETH/BTC ratio
Checking whether log(ETH/BTC) mean-reverts (cointegration).

In [ ]:
log_ratio = daily_log_price["ETHUSDT"] - daily_log_price["BTCUSDT"]
ratio_ret = log_ratio.diff().dropna()

print("ADF p-value on log(ETH/BTC):", round(adfuller(log_ratio, autolag="AIC")[1], 3))
print("Engle-Granger cointegration p-value:", round(coint(daily_log_price["ETHUSDT"], daily_log_price["BTCUSDT"])[1], 3))
for year in [2021, 2022, 2023]:
    print(f"  {year}: ADF p on log ratio = {adfuller(log_ratio.loc[str(year)], autolag='AIC')[1]:.3f}")
pd.DataFrame({q: variance_ratio(ratio_ret, q) for q in [2, 5, 10, 20]}, index=["VR", "z"]).T

### 11.6 How much of ETH is just BTC?
R² of ETH on BTC at each horizon, and what's left over.

In [ ]:
rows = {}
for h in ["1h", "1d"]:
    r = rets[h]
    beta = r["BTCUSDT"].cov(r["ETHUSDT"]) / r["BTCUSDT"].var()
    resid = r["ETHUSDT"] - beta * r["BTCUSDT"]
    rows[h] = {
        "beta": beta,
        "R² (share explained by BTC)": r["BTCUSDT"].corr(r["ETHUSDT"]) ** 2,
        "residual ann. vol %": resid.std() * np.sqrt(PER_YEAR[h]) * 100,
        "residual lag-1 autocorr": resid.autocorr(1),
    }
pd.DataFrame(rows)

### Observations: BTC and ETH together
- **Correlation** 0.85 (1h), 0.85 (4h), 0.82 (1d), 0.83 (1w). BTC explains 73% (hourly) / 68% (daily) of ETH's variance, beta ≈ 1.08. ETH's leftover part has ~44–48% vol and ≈ 0 autocorrelation
- **Rolling 30-day:** correlation 0.68–0.95 (mean 0.87), beta 0.71–1.41 (mean 1.07)
- **No lead-lag hourly:** ±1 h correlations 0.002 / 0.006, inside ±2 s.e. (0.012). Small −0.02 at ±2 h in both directions
- **Crash together, rally apart:** worst 1% of days, ETH also in its worst 1% 82% of the time (a normal pair would give 41%). Best 1%: only 27% (normal 42%). Hourly: 65% vs 46% for crashes
- **ETH/BTC ratio:** full-sample ADF p = 0.017, Engle-Granger p = 0.054, but within each year ADF p = 0.48 / 0.20 / 0.69. The full-sample result comes from one move up (2021) and back (2022–23)

## 12. Predictive information

Which variables known at the close of hour *t* say anything about the return from *t* to *t + h*?

How I tested it:
- predictors use data up to and including hour *t*
- targets: forward log returns over 1 h, 4 h, 24 h
- IC = Spearman correlation between predictor and target
- for horizon *h* I keep every *h*-th observation so targets don't overlap (t ≈ IC·√n)
- IC also per year, to see if the sign holds

In [ ]:
def predictors(df, other):
    lp = np.log(df["close"])
    r = lp.diff()
    X = pd.DataFrame(index=df.index)
    for h in [1, 4, 24, 168, 720]:
        X[f"mom_{h}h"] = lp - lp.shift(h)
    X["rv_24h"] = np.sqrt((r ** 2).rolling(24).sum())
    X["rv_ratio_24h_30d"] = X["rv_24h"] / np.sqrt((r ** 2).rolling(720).sum() / 30)
    log_vol = np.log(df["quote_volume"] + 1)
    X["volume_surprise"] = log_vol - log_vol.rolling(168).mean()
    X["taker_ratio_1h"] = df["taker_buy_base"] / df["volume"]
    X["taker_ratio_24h"] = df["taker_buy_base"].rolling(24).sum() / df["volume"].rolling(24).sum()
    X["range_1h"] = np.log(df["high"] / df["low"])
    X["dist_ma_200h"] = lp - lp.rolling(200).mean()
    X["other_ret_1h"] = np.log(other["close"]).diff()
    return X


def targets(df, horizons=(1, 4, 24)):
    lp = np.log(df["close"])
    return pd.DataFrame({f"fwd_{h}h": lp.shift(-h) - lp for h in horizons})

In [ ]:
def ic_table(X, Y, absolute=False):
    rows = []
    for target in Y:
        h = int(target.split("_")[1][:-1])
        y = Y[target].abs() if absolute else Y[target]
        for feature in X:
            d = pd.concat([X[feature], y], axis=1).dropna().iloc[::h]      # non-overlapping
            ic = d.corr(method="spearman").iloc[0, 1]
            by_year = d.groupby(d.index.year).apply(lambda g: g.corr(method="spearman").iloc[0, 1])
            rows.append({
                "feature": feature, "target": target, "IC": ic, "t": ic * np.sqrt(len(d)),
                **{str(y_): v for y_, v in by_year.items()},
                "same sign every year": bool((np.sign(by_year) == np.sign(ic)).all()),
            })
    return pd.DataFrame(rows)


X = {s: predictors(train[s], train[o]) for s, o in [("BTCUSDT", "ETHUSDT"), ("ETHUSDT", "BTCUSDT")]}
Y = {s: targets(train[s]) for s in ASSETS}

### 12.1 Direction
Sorted by |t|.

In [ ]:
ic_direction = {s: ic_table(X[s], Y[s]) for s in ASSETS}
ic_direction["BTCUSDT"].sort_values("t", key=abs, ascending=False).head(12)

In [ ]:
ic_direction["ETHUSDT"].sort_values("t", key=abs, ascending=False).head(12)

**Multiple testing check:** with this many tests some will look significant by chance.

In [ ]:
all_ic = pd.concat(ic_direction, names=["asset"]).reset_index(level=0)
n_tests = len(all_ic)
pd.Series({
    "tests run": n_tests,
    "|t| > 2 observed": (all_ic["t"].abs() > 2).sum(),
    "|t| > 2 expected by chance": round(n_tests * 0.0455, 1),
    "|t| > 3 observed": (all_ic["t"].abs() > 3).sum(),
    "|t| > 3 expected by chance": round(n_tests * 0.0027, 2),
    "|t| > 2 AND same sign all 3 years": ((all_ic["t"].abs() > 2) & all_ic["same sign every year"]).sum(),
})

### 12.2 Magnitude
Same test, target = |forward return|.

In [ ]:
ic_magnitude = {s: ic_table(X[s], Y[s], absolute=True) for s in ASSETS}
pd.concat(ic_magnitude, names=["asset"]).reset_index(level=0) \
    .query("target == 'fwd_24h'").sort_values("t", key=abs, ascending=False).head(12)

### 12.3 Forward return by predictor decile
Correlation only catches monotonic effects, so looking at deciles too. Plotting the four strongest direction candidates. Shaded band = ±15 bps (one side of the 0.15% cost; a round trip is 30 bps).

In [ ]:
def decile_curve(x, y, h):
    d = pd.concat([x, y], axis=1).dropna().iloc[::h]
    d.columns = ["x", "y"]
    d["decile"] = pd.qcut(d["x"], 10, labels=False, duplicates="drop") + 1
    return d.groupby("decile")["y"].mean() * 1e4          # in bps


top = all_ic.sort_values("t", key=abs, ascending=False).head(4)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), sharey=False)
for ax, (_, row) in zip(axes, top.iterrows()):
    h = int(row["target"].split("_")[1][:-1])
    curve = decile_curve(X[row["asset"]][row["feature"]], Y[row["asset"]][row["target"]], h)
    ax.bar(curve.index, curve.values, color=COLORS[row["asset"]])
    ax.axhspan(-15, 15, color="grey", alpha=0.2)
    ax.set_title(f"{row['asset'][:3]} {row['feature']}\n→ {row['target']} (IC {row['IC']:.3f})", fontsize=9)
    ax.set_xlabel("predictor decile")
axes[0].set_ylabel("mean forward return (bps)")
fig.tight_layout()
savefig(fig, "decile_curves")

### 12.4 Significant vs big enough
For the strongest reversal signals: bps between the extreme deciles vs the 30 bps a round trip costs.

In [ ]:
rows = {}
for s in ASSETS:
    for feature, target, h in [("mom_1h", "fwd_1h", 1), ("mom_4h", "fwd_1h", 1), ("mom_4h", "fwd_4h", 4), ("taker_ratio_1h", "fwd_1h", 1)]:
        curve = decile_curve(X[s][feature], Y[s][target], h)
        rows[(s, feature, target)] = {
            "lowest decile (bps)": curve.iloc[0],
            "highest decile (bps)": curve.iloc[-1],
            "spread D1 - D10 (bps)": curve.iloc[0] - curve.iloc[-1],
            "round-trip cost (bps)": 30,
        }
pd.DataFrame(rows).T

### 12.5 Do taker flow and the other coin add anything beyond the coin's own return?
Partial IC: removing the part explained by own 1 h and 4 h returns, then correlating what's left.

In [ ]:
rows = {}
for s in ASSETS:
    f = pd.concat([X[s][["mom_1h", "mom_4h", "taker_ratio_1h", "other_ret_1h"]], Y[s]["fwd_1h"]], axis=1).dropna().rank()
    controls = sm.add_constant(f[["mom_1h", "mom_4h"]])
    y_resid = sm.OLS(f["fwd_1h"], controls).fit().resid
    for feature in ["taker_ratio_1h", "other_ret_1h"]:
        x_resid = sm.OLS(f[feature], controls).fit().resid
        ic = np.corrcoef(x_resid, y_resid)[0, 1]
        rows[(s, feature)] = {"partial IC (fwd_1h)": ic, "t": ic * np.sqrt(len(f))}
pd.DataFrame(rows).T

### 12.6 Daily time-series momentum
Long if the past N-day return is positive, short if negative, next-day return before costs. Checking neighbouring lookbacks too.

In [ ]:
rows = {}
for s in ASSETS:
    r = rets["1d"][s]
    for n in [5, 10, 14, 20, 30, 45, 60, 90, 120]:
        signal_ret = (np.sign(r.rolling(n).sum()) * r.shift(-1)).dropna()
        by_year = signal_ret.groupby(signal_ret.index.year).mean() * 365 * 100
        rows[(s, n)] = {
            **{str(k): v for k, v in by_year.items()},
            "all (ann. %)": signal_ret.mean() * 365 * 100,
            "t-stat": signal_ret.mean() / signal_ret.std() * np.sqrt(len(signal_ret)),
        }
tsmom = pd.DataFrame(rows).T
tsmom.index.names = ["asset", "lookback (days)"]
tsmom

### Observations: predictive information
- **Short-term reversal:** 33 of 78 tests have |t| > 2 (3.5 expected by chance), 17 have |t| > 3 (0.2 expected), 30 keep the same sign all 3 years. Strongest: last 4 h return → next 1 h, IC −0.070 (BTC, t = −11.3) and −0.054 (ETH, t = −8.7)
- **Size of it:** extreme-decile spread only ~2–4 bps over 1 h and ~13–15 bps over 4 h, vs 30 bps per round trip
- **U shape:** for the last 1 h return both extreme deciles are followed by slightly positive returns
- **Partial IC:** after own returns, taker ratio keeps −0.016 (t = −2.6) for BTC only; BTC's return keeps −0.021 (t = −3.5) for ETH only. For ETH the taker-ratio deciles go the opposite way to its IC (top +4 bps, bottom −1 bps)
- **Size is predictable:** last hour's range and past 24 h vol predict the size of the next 24 h move with IC 0.29–0.39, same sign every year. Volume surprise ranks well overall (0.10–0.12) but flips sign between years
- **Daily momentum:** 30-day lookback gives BTC +64% a year (t = 1.7), but 20 / 45 days give +33% / +26% and 90 days −3.5%. No lookback reaches |t| = 2 for either coin

## 13. Summary

Full write-up: `reports/data_research_report.md`.

| # | Observation | Where |
|---|---|---|
| 1 | Data is clean; 7 outages (16 flagged hours), 2 empty candles found | 1–2 |
| 2 | BTC trades/day ×3.8 during the zero-fee period while ETH fell | 2.6 |
| 3 | Fat tails (α ≈ 3, kurtosis 13–16 hourly); biggest jumps come from calm markets | 4 |
| 4 | Volatility is persistent and forecastable out of sample (HAR R² 0.60 / 0.73 on 2023) | 7 |
| 5 | Intraday vol cycle: 14–15 UTC high, 04–06 UTC low, same every year | 8 |
| 6 | No hour or weekday *return* effect | 8 |
| 7 | Direction ≈ random overall; VR flips by year (0.83 → 1.34 for BTC) | 6 |
| 8 | 1–4 h reversal is statistically real (t ≈ −11) but smaller than 30 bps costs | 12 |
| 9 | BTC–ETH correlation ~0.85; crash together 82%, rally together 27% | 11 |
| 10 | ETH/BTC "mean reversion" only on the full sample, not within any year | 11.5 |
| 11 | Daily momentum results jump around between neighbouring lookbacks | 12.6 |
| 12 | Below the 200-day MA: same next-day return, higher vol | 9.4 |